# Bratteli diagrams and dimension truncation

Companion to Figs. 1–3 and Sects. 5.1, 5.3–5.4 of Grinko and Ozols, *Linear Programming with Unitary-Equivariant Constraints* ([paper](https://doi.org/10.1007/s00220-024-05108-1), [repository guide](README.md)).
The notebook constructs the branching graph for
$B_{0,0} \to B_{1,0} \to \cdots \to B_{p,0} \to B_{p,1} \to \cdots \to B_{p,q}$,
labels each edge by its content at $\delta=d$, and marks the vertices and edges removed for the matrix algebras $A^d_{p,q}$.
The default is **$(p,q,d)=(2,2,2)$**, corresponding to the dimension-two case in Fig. 3.

Run the cells in order in a **SageMath kernel**.
The construction and counts need SageMath only; the final export section additionally needs the drawing tools listed below.
This notebook does not read or overwrite the idempotent database.

## 1. Choose the figure parameters

| Paper figure | `p` | `q` | `d` | Interpretation |
|---|---:|---:|---:|---|
| Fig. 1 | 4 | 0 | 2 or 3 | Young's lattice through level 4, with the corresponding dimension cutoff |
| Fig. 2 | 2 | 2 | 4 or greater | Full walled Brauer branching graph; no red edges |
| Fig. 3 | 2 | 2 | 2 or 3 | Walled Brauer branching graph with the matrix-algebra truncation |

Changing `d` changes the red-edge cutoff and the numeric content labels, but not the full branching graph.
Change the parameters below, then rerun every subsequent cell.
The export directory is relative to the notebook kernel's working directory; rerunning the same parameters replaces only the matching figure files there.

In [ ]:
from pathlib import Path

p = 2
q = 2
d = 2
output_dir = Path("generated") / "bratteli_figures"

if p not in ZZ or q not in ZZ or p < 0 or q < 0:
    raise ValueError("p and q must be nonnegative integers")
if d not in ZZ or d < 1:
    raise ValueError("d must be a positive integer")

## 2. Build the branching graph

A vertex is `(lambda_l, lambda_r, level)`, with a bipartition as in eq. (84).
The level distinguishes identical bipartitions at different points of the tower; for example, $(\emptyset,\emptyset)$ occurs at both levels 0 and 4 when $p=q=2$.
The cover rule is the branching rule of Sect. 5.1: first add a cell to $\lambda^l$; after the wall, either add a cell to $\lambda^r$ or remove one from $\lambda^l$.
`wba_covers` is applied to the allowed vertices generated below.

In [ ]:
def wba_covers(source, target):
    """Whether two allowed vertices are consecutive in the branching graph."""
    source_left, source_right, source_level = source
    target_left, target_right, target_level = target

    if source_level + 1 != target_level:
        return False

    # Before the wall: add one cell to the left partition.
    if source_left.size() == source_level and target_left.size() == target_level:
        return target_left.contains(source_left)

    # After the wall: add on the right, or remove on the left.
    if source_left.size() <= source_level and target_left.size() < target_level:
        if target_left.size() == source_left.size():
            return (target_right.contains(source_right)
                    and source_left.contains(target_left))
        if target_right.size() == source_right.size():
            return (source_left.contains(target_left)
                    and source_right.contains(target_right))
    return False

In [3]:
elements_up_to_p = [
    (left, Partition([]), level)
    for level in range(p + 1)
    for left in Partitions(level)
]
elements_after_p = [
    (left, right, p + right_level)
    for right_level in range(1, q + 1)
    for contractions in range(min(p, right_level), -1, -1)
    for left in Partitions(p - contractions)
    for right in Partitions(right_level - contractions)
]

elements = elements_up_to_p + elements_after_p
Y = Poset((elements, wba_covers), facade=True)
H = Y.hasse_diagram().copy()
print(f"Full graph: {H.order()} vertices, {H.size()} edges.")

Full graph: 13 vertices, 15 edges.


## 3. Label contents and mark the dimension cutoff

For a cell in row $i$ and column $j$, Sect. 5.3 assigns content $j-i$ when it is added on the left, $i-j$ when it is removed on the left, and $j-i+\delta$ when it is added on the right.
Sage numbers rows and columns from zero, which leaves these differences unchanged.
The plot substitutes $\delta=d$.

A vertex survives the truncation of Sect. 5.4 when
$\operatorname{len}(\lambda^l)+\operatorname{len}(\lambda^r)\le d$.
An edge is **red** if either endpoint fails this test; otherwise it is **black**.
Keep the surviving vertices and black edges to obtain the Bratteli diagram of the matrix-algebra tower.
The draft retains the removed part in red to make the cutoff visible.

In [4]:
def color_edge(edge, d):
    """Mark an edge red if either endpoint fails the dimension cutoff."""
    source, target = edge[:2]
    source_rows = source[0].length() + source[1].length()
    target_rows = target[0].length() + target[1].length()
    return "red" if source_rows > d or target_rows > d else "black"


def compute_content(edge, p, z=var("delta")):
    """Return the same edge labelled by its Jucys--Murphy content."""
    source, target = edge[:2]
    left_change = Set(target[0].cells()).symmetric_difference(Set(source[0].cells()))

    if target[2] <= p:
        row, column = left_change[0]
        content = column - row
    elif len(left_change) == 1:
        row, column = left_change[0]
        content = row - column
    else:
        right_change = Set(target[1].cells()).symmetric_difference(Set(source[1].cells()))
        row, column = right_change[0]
        content = column - row + z
    return (source, target, content)


# H is a separate graph, so Y remains the unlabelled branching poset.
for edge in H.edges(sort=False):
    H.delete_edge(edge)
    H.add_edge(compute_content(edge, p, d))

edge_colors = {edge: color_edge(edge, d) for edge in H.edges(sort=False)}
retained_vertices = [
    vertex for vertex in elements
    if vertex[0].length() + vertex[1].length() <= d
]
retained_edges = sum(color == "black" for color in edge_colors.values())
print(f"For d = {d}: {len(retained_vertices)} retained vertices, "
      f"{retained_edges} retained edges, {H.size() - retained_edges} red edges.")

For d = 2: 9 retained vertices, 10 retained edges, 5 red edges.


## 4. Export editable drafts and PDFs

These are working drafts of the paper's figures.
The published artwork additionally has column headers, vertex labels without the level, `d=2`/`d=3` annotations instead of red edges, and the highlighted path in Fig. 3; it omits the content labels shown here.
The full-poset export shows the branching structure without content labels or truncation colours.
The content export shows the same graph with the numeric contents and red/black edges.
Neither export removes vertices from the full graph.

**Rendering requirements.** Use SageMath 9.6 or later for `sage.misc.latex_standalone.TikzPicture`, plus `dot2tex`, Graphviz, and a LaTeX installation with `tikz`, `tkz-graph` and `tkz-berge`.
The graph format is explicitly `dot2tex`, so a missing dependency is reported instead of silently falling back to a layout that loses colours and labels.
On SageMath 9.4/9.5 the older alternative is `from slabbe import TikzPicture`, with the optional `slabbe` package installed.

**Tested compatibility.** Native SageMath 10.9 with `dot2tex==2.11.3` and `pyparsing==3.3.3` failed during rendering with `TypeError: sequence item 0: expected str instance, ParseResults found` in `dot2tex.dotparsing.parse_html`.
The same notebook and both PDF exports succeeded with `pyparsing==3.1.4` supplied in an isolated environment.
If that parser error occurs, use the tested `dot2tex==2.11.3` / `pyparsing==3.1.4` combination in the Sage environment running this kernel, then restart the kernel.
No package installation or downgrade is performed by this notebook.

The cells below save two standalone `.tex` sources and two PDFs in `output_dir`, print their absolute paths, and provide download links.
PDF generation uses `view=False` so a batch run does not open external viewer windows.

In [ ]:
from IPython.display import FileLink, display
from sage.misc.latex_standalone import TikzPicture

latex.extra_preamble(
    "\\usepackage{tikz}\n"
    "\\usepackage{tkz-graph}\n"
    "\\usepackage{tkz-berge}\n"
    "\\usetikzlibrary{arrows,shapes}"
)
latex.add_to_preamble("\\renewcommand{\\emptyset}{\\varnothing}")

H.latex_options().set_options(
    format="dot2tex",
    edge_colors=edge_colors,
    edge_labels=True,
    tkz_style="Art",
    rankdir="right",
)

poset_picture = TikzPicture.from_poset(Y, rankdir="right", edge_labels=False)
content_picture = TikzPicture.from_graph(
    H, rankdir="right", edge_labels=True, edge_colors=edge_colors,
)

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
poset_stem = output_dir / f"bratteli_{p}_{q}_full"
content_stem = output_dir / f"bratteli_{p}_{q}_d{d}_contents"

poset_picture.tex(filename=str(poset_stem.with_suffix(".tex")))
content_picture.tex(filename=str(content_stem.with_suffix(".tex")))
path_to_file = poset_picture.pdf(
    filename=str(poset_stem.with_suffix(".pdf")), view=False,
)
path_to_file_h = content_picture.pdf(
    filename=str(content_stem.with_suffix(".pdf")), view=False,
)

for stem in (poset_stem, content_stem):
    for suffix in (".pdf", ".tex"):
        artifact = stem.with_suffix(suffix)
        print(artifact.resolve())
        display(FileLink(str(artifact)))